In [ ]:
from itertools import product

import matplotlib.pyplot as plt
import pandas as pd

from openplaces.api import get_admin2, get_admin_ids
from openplaces.io import read_parquet
from openplaces.path import cache_path
from openplaces.recipe import get_recipe_by_id

DATASETS = {
    'parcels': 'US-NC_parcel-nconemap-2025',
    'nsi': 'US_building-usace-2022',
    'fema': 'US_building-fema-2023',
    'microsoft': 'US_building-microsoft-v2',
}


def load_entities(admin_id, recipe, geom=False):
    if isinstance(recipe, str):
        recipe = get_recipe_by_id(recipe)
    parquet_path = cache_path(admin_id, recipe['entity'])
    return read_parquet(parquet_path, geom=geom)


def get_stats(gdf):
    gdf_stats = (gdf.notnull() & gdf.ne('')).sum().rename('n_values').to_frame()
    gdf_stats['frac_values'] = gdf_stats['n_values'] / len(gdf)
    gdf_stats.index.name = 'variable'
    gdf_stats = gdf_stats.join(gdf.describe(percentiles=[0.5]).T.drop(columns='count'))
    return gdf_stats

In [ ]:
admin_ids = get_admin_ids(2, 'US-NC')

# Compute stats

In [ ]:
stats_lists = {k: [] for k in DATASETS.keys()}
for admin_id, (dataset_key, dataset_recipe) in product(admin_ids, DATASETS.items()):
    print('.', end='')
    dataset = load_entities(admin_id, dataset_recipe)
    dataset_stats = get_stats(dataset)
    dataset_stats.insert(0, 'admin_id', admin_id)
    stats_lists[dataset_key] += [dataset_stats]

stats = {
    dataset_key: pd.concat(stats_lists[dataset_key])
    .reset_index()
    .set_index(['variable', 'admin_id'])
    for dataset_key in DATASETS.keys()
}

# Map variable presence

## By variable

In [ ]:
for dataset_key in DATASETS.keys():
    data = (
        stats[dataset_key]
        .groupby('variable')['frac_values']
        .mean()
        .mul(100)
        .loc[stats[dataset_key].index.get_level_values(0).unique()][::-1]
    )

    fig, ax = plt.subplots(figsize=(3, len(data) * 0.2))
    data.plot(kind='barh', ax=ax)
    ax.set_xlim(0, 100)
    ax.grid(axis='x', color='black', linewidth=0.2)
    ax.set_xlabel('average % non-null values (county)')
    ax.set_ylabel(None)
    ax.set_title(dataset_key)
    plt.plot()

## By county

In [ ]:
admin2 = get_admin2(
    'US-NC', geom=True, recipe=get_recipe_by_id('US_admin-nhgis-2020_admin2')
)

In [ ]:
import matplotlib.pyplot as plt

BINS = [0, 0.01, 0.03, 0.1, 0.2, 0.5, 0.8, 0.9, 0.97, 0.99, 1]

for dataset_key in DATASETS.keys():
    print(dataset_key)

    for variable in stats[dataset_key].index.get_level_values(0).unique():

        data = 1 - stats[dataset_key].loc[variable]['frac_values']
        if data.eq(0).all():
            continue

        fig, ax = plt.subplots(figsize=(9, 3))
        admin2.join(data).plot(
            'frac_values',
            ax=ax,
            scheme='user_defined',
            classification_kwds={'bins': BINS[1:], 'lowest': BINS[0]},
            legend=True,
            legend_kwds={
                'loc': 'center left',
                'bbox_to_anchor': (1, 0.5),
                'title': '% empty',
                'labels': [
                    f'{int(BINS[i]*100)} - {int(BINS[i+1]*100)}'
                    for i in range(len(BINS) - 1)
                ],
            },
            cmap='RdYlBu_r',
        )
        ax.set_title(f'{dataset_key}: {variable}')
        ax.axis('off')
        plt.show()